In [39]:

import requests
import pandas as pd

url = "https://api.census.gov/data/2024/acs/acs5"
params = {
    "get": "NAME,B19013_001E,B27001_001E,B01003_001E", #name,median household income, health insurance coverage, total population
    "for": "county:*",          # all counties
    "in": "state:*",            # all states
    "key": '750defc8aa4a25457ddbe3a4f3b4bf27e31595b9'
}
r = requests.get(url, params=params)
df = pd.DataFrame(r.json()[1:], columns=r.json()[0])

In [40]:
df.to_csv("census_county_data_raw.csv", index=False)

In [41]:
# Create a combined FIPS code column for merging with other datasets later
df["geoid"] = df["state"] + df["county"]
df["geoid"] = df["state"].str.zfill(2) + df["county"].str.zfill(3)

# Convert numeric columns from strings (Census API returns everything as strings)
df["B19013_001E"] = pd.to_numeric(df["B19013_001E"], errors="coerce")
df["B27001_001E"] = pd.to_numeric(df["B27001_001E"], errors="coerce")
df["B01003_001E"] = pd.to_numeric(df["B01003_001E"], errors="coerce")

# Rename to something readable
df = df.rename(columns={
    "B19013_001E": "median_household_income",
    "B27001_001E": "health_insurance_total",
    "B01003_001E": "total_population"
})


In [42]:
df.to_csv("census_county_data.csv", index=False)

In [43]:
import requests
import zipfile
import io
import pandas as pd

# Download and unzip in memory — no need to save the zip file
url = "https://www2.census.gov/programs-surveys/sahie/datasets/time-series/estimates-acs/sahie-2022-csv.zip"

response = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(response.content))

# Check what's inside the zip
print(z.namelist())


['sahie_2022.csv']


In [44]:
# Read the raw lines to find where the actual data starts
with z.open('sahie_2022.csv') as f:
    lines = f.readlines()

# Print the first 85 lines to find where the header row is
for i, line in enumerate(lines[:85]):
    print(i, line.decode('utf-8').strip())

0 Filename:  sahie_2022.csv
1 Created:   12JUL24  02:44
2 Description:
3 
4 Model-based Small Area Health Insurance Estimates (SAHIE) for Counties and States, 2022
5 Source: U.S. Census Bureau, Small Area Health Insurance Estimates (SAHIE) Program, July 2024, Project No. P-6000007 / Approval CBDRB FY24-0310
6 
7 NOTE: VERY LONG .CSV FILES NOT ALWAYS IMPORTED INTO EXCEL CORRECTLY - CHECK FOR TRUNCATION
8 
9 File Layout and Definitions:
10 
11 Variable      Format      Description
12 year            4      Year of Estimate
13 version         8      Release Version
14 Blank   : YEAR other than 2013, Only Version
15 Original: 2013 only, Original Version
16 Updated : 2013 only, Updated Version (May 2016)
17 statefips       2      Unique FIPS code for each state
18 countyfips      3      Unique FIPS code for each county within a state
19 geocat          2      Geography category
20 40 - State geographic identifier
21 50 - County geographic identifier
22 agecat          1      Age category
23

In [45]:
df_sahie = pd.read_csv(z.open('sahie_2022.csv'), skiprows=84)
for col in ["PCTUI", "NUI", "NIPR", "statefips", "countyfips", "agecat", "iprcat", "sexcat", "racecat"]:
    df_sahie[col] = pd.to_numeric(df_sahie[col], errors="coerce")

C:\Users\17063\AppData\Local\Temp\ipykernel_46484\4102004643.py:1: DtypeWarning: Columns (0: NIPR, 1: nipr_moe, 2: NUI, 3: nui_moe, 4: NIC, 5: nic_moe, 6: PCTUI, 7: pctui_moe, 8: PCTIC, 9: pctic_moe, 10: PCTELIG, 11: pctelig_moe, 12: PCTLIIC, 13: pctliic_moe) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sahie = pd.read_csv(z.open('sahie_2022.csv'), skiprows=84)


In [46]:
df_sahie.to_csv("sahie_county_data_raw.csv", index=False)

In [47]:
print(df_sahie.columns.tolist())

['year', 'version', 'statefips', 'countyfips', 'geocat', 'agecat', 'racecat', 'sexcat', 'iprcat', 'NIPR', 'nipr_moe', 'NUI', 'nui_moe', 'NIC', 'nic_moe', 'PCTUI', 'pctui_moe', 'PCTIC', 'pctic_moe', 'PCTELIG', 'pctelig_moe', 'PCTLIIC', 'pctliic_moe', 'state_name', 'county_name', 'Unnamed: 25']


In [48]:
# Filter to all ages, all incomes, both sexes, all races
df_sahie = df_sahie[
    (df_sahie["agecat"] == 0) &
    (df_sahie["iprcat"] == 0) &
    (df_sahie["sexcat"] == 0) &
    (df_sahie["racecat"] == 0)
]

# Build geoid and uninsured rate
df_sahie["geoid"] = df_sahie["statefips"].astype(str).str.zfill(2) + \
                    df_sahie["countyfips"].astype(str).str.zfill(3)
df_sahie["uninsured_rate"] = df_sahie["PCTUI"] / 100

# Drop statewide rows (countyfips == 0)
df_sahie = df_sahie[df_sahie["countyfips"] != 0]

In [49]:
df_sahie = df_sahie[["geoid", "uninsured_rate", "NUI", "NIPR"]]

In [50]:
df_sahie = df_sahie.rename(columns={
    "NUI": "uninsured_count",
    "NIPR": "total_population"
})

In [51]:
df_sahie.to_csv("sahie_county_data.csv", index=False)

In [52]:
# HPSA file
url = "https://data.hrsa.gov/DataDownload/DD_Files/BCD_HPSA_FCT_DET_PC.csv"
response = requests.get(url)

df_hpsa = pd.read_csv(io.BytesIO(response.content))

C:\Users\17063\AppData\Local\Temp\ipykernel_46484\3084611216.py:5: DtypeWarning: Columns (0: BHCMIS Organization Identification Number) have mixed types. Specify dtype option on import or set low_memory=False.
  df_hpsa = pd.read_csv(io.BytesIO(response.content))


In [53]:
print(df_hpsa.shape)
print(df_hpsa.columns.tolist())

(74920, 66)
['HPSA Name', 'HPSA ID', 'Designation Type', 'HPSA Discipline Class', 'HPSA Score', 'PC MCTA Score', 'Primary State Abbreviation', 'HPSA Status', 'HPSA Designation Date', 'HPSA Designation Last Update Date', 'Metropolitan Indicator', 'HPSA Geography Identification Number', 'HPSA Degree of Shortage', 'Withdrawn Date', 'HPSA FTE', 'HPSA Designation Population', '% of Population Below 100% Poverty', 'HPSA Formal Ratio', 'HPSA Population Type', 'Rural Status', 'Longitude', 'Latitude', 'BHCMIS Organization Identification Number', 'Break in Designation', 'Common County Name', 'Common Postal Code', 'Common Region Name', 'Common State Abbreviation', 'Common State County FIPS Code', 'Common State FIPS Code', 'Common State Name', 'County Equivalent Name', 'County or County Equivalent Federal Information Processing Standard Code', 'Discipline Class Number', 'HPSA Address', 'HPSA City', 'HPSA Component Name', 'HPSA Component Source Identification Number', 'HPSA Component State Abbrevia

In [55]:
df_hpsa.to_csv("hpsa_county_data_raw.csv", index=False)

In [ ]:
# Filter to active primary care designations
df_hpsa = df_hpsa[df_hpsa["HPSA Status"] == "Designated"]
df_hpsa = df_hpsa[df_hpsa["HPSA Discipline Class"] == "Primary Care"]

# Keep useful columns
df_hpsa = df_hpsa[[
    "Common State FIPS Code",
    "State and County Federal Information Processing Standard Code",
    "Common County Name",
    "Common State Name",
    "HPSA Name",
    "HPSA Status",
    "HPSA Score",
    "Designation Type",
    "HPSA Designation Population",
    "HPSA Estimated Underserved Population",
    "HPSA Formal Ratio",        # provider-to-population ratio
    "% of Population Below 100% Poverty",
    "Rural Status",
    "Metropolitan Indicator"
]]

# The full 5-digit FIPS is already combined in one column
df_hpsa = df_hpsa.rename(columns={
    "State and County Federal Information Processing Standard Code": "geoid",
    "HPSA Score": "hpsa_score",
    "HPSA Formal Ratio": "provider_ratio",
    "% of Population Below 100% Poverty": "pct_below_poverty",
    "HPSA Designation Population": "hpsa_population",
    "HPSA Estimated Underserved Population": "underserved_population",
    "Rural Status": "rural_status",
    "Metropolitan Indicator": "metro_indicator",
    "Common County Name": "county_name",
    "Common State Name": "state_name"
})

# Zero-pad geoid just in case
df_hpsa["geoid"] = df_hpsa["geoid"].astype(str).str.zfill(5)

# Aggregate to one row per county
df_hpsa_county = df_hpsa.groupby("geoid").agg(
    hpsa_score_max=("hpsa_score", "max"), #composite HPSA score.
    hpsa_designation_count=("HPSA Name", "count"), #how many shortage areas in a county
    pct_below_poverty=("pct_below_poverty", "max")
).reset_index()
df_hpsa_county["hpsa_designated"] = (df_hpsa_county["hpsa_score_max"] > 0).astype(int)

In [61]:
df_hpsa.to_csv("hpsa_county_data.csv", index=False)
df_hpsa_county.to_csv("hpsa_county_composite_score.csv", index=False)

In [63]:
url = "https://data.hrsa.gov/DataDownload/DD_Files/UDS_2023_Data.zip"
response = requests.get(url)

# Check what you actually got back
print(response.headers.get("Content-Type"))
print(response.status_code)

text/html
404


In [7]:
import requests
import pandas as pd

url = "https://data.cdc.gov/resource/swc5-untb.json"

all_records = []
limit = 50000
offset = 0

while True:
    params = {
        "$limit": limit,
        "$offset": offset
    }
    
    response = requests.get(url, params=params)
    batch = response.json()
    
    if not batch:
        break
        
    all_records.extend(batch)
    print(f"Fetched {len(all_records)} records so far...")
    
    if len(batch) < limit:
        break
        
    offset += limit

df_places = pd.DataFrame(all_records)
print(f"\nTotal records: {df_places.shape}")
print(df_places.columns.tolist())

Fetched 50000 records so far...
Fetched 100000 records so far...
Fetched 150000 records so far...
Fetched 200000 records so far...
Fetched 229298 records so far...

Total records: (229298, 24)
['year', 'stateabbr', 'statedesc', 'locationname', 'datasource', 'category', 'measure', 'data_value_unit', 'data_value_type', 'data_value', 'low_confidence_limit', 'high_confidence_limit', 'totalpopulation', 'totalpop18plus', 'locationid', 'categoryid', 'measureid', 'datavaluetypeid', 'short_question_text', 'geolocation', ':@computed_region_hjsp_umg2', ':@computed_region_skr5_azej', 'data_value_footnote_symbol', 'data_value_footnote']


In [8]:
df_places.to_csv("cdc_prevalence_raw.csv", index=False)

In [9]:
# See what health measures are available
print(df_places["measureid"].unique())

<StringArray>
[   'ARTHRITIS',      'CASTHMA',       'STROKE',   'DISABILITY',
      'OBESITY',     'DIABETES',   'DEPRESSION',      'HEARING',
        'BINGE',       'BPHIGH',       'VISION',     'MOBILITY',
     'SELFCARE',        'MHLTH',     'HIGHCHOL',        'PHLTH',
 'COLON_SCREEN',          'LPA',         'COPD',    'INDEPLIVE',
          'CHD',     'MAMMOUSE',      'CHECKUP',        'BPMED',
     'CSMOKING',   'HOUSINSECU',  'SHUTUTILITY',   'FOODINSECU',
    'TEETHLOST',        'GHLTH',     'LACKTRPT',       'CANCER',
   'EMOTIONSPT',       'DENTAL',   'LONELINESS',    'FOODSTAMP',
      'ACCESS2',    'COGNITION',   'CHOLSCREEN',        'SLEEP']
Length: 40, dtype: str


In [10]:
measures = [
    "ACCESS2",   # lack of health insurance (redundant with SAHIE but good validation)
    "DIABETES",  # diabetes prevalence
    "HIGHCHOL",  # high cholesterol
    "BPHIGH",    # high blood pressure
    "CASTHMA",   # asthma
    "CHECKUP",   # no routine checkup in past year — very relevant for care desert validation
    "DENTAL",    # no dental visit
]

df_places = df_places[df_places["measureid"].isin(measures)]

In [11]:
print(df_places.shape)        # how many rows and columns
print(df_places.columns.tolist())  # column names
print(df_places.head())       # first few rows
print(df_places["measureid"].value_counts())  # what health measures are in there

(41786, 24)
['year', 'stateabbr', 'statedesc', 'locationname', 'datasource', 'category', 'measure', 'data_value_unit', 'data_value_type', 'data_value', 'low_confidence_limit', 'high_confidence_limit', 'totalpopulation', 'totalpop18plus', 'locationid', 'categoryid', 'measureid', 'datavaluetypeid', 'short_question_text', 'geolocation', ':@computed_region_hjsp_umg2', ':@computed_region_skr5_azej', 'data_value_footnote_symbol', 'data_value_footnote']
    year stateabbr statedesc locationname datasource         category  \
1   2023        AR  Arkansas       Fulton      BRFSS  Health Outcomes   
9   2023        CO  Colorado         Lake      BRFSS  Health Outcomes   
13  2023        FL   Florida      Brevard      BRFSS  Health Outcomes   
14  2023        FL   Florida       Citrus      BRFSS  Health Outcomes   
57  2023        KS    Kansas      Haskell      BRFSS  Health Outcomes   

                            measure data_value_unit   data_value_type  \
1       Current asthma among adults  

In [12]:
# Keep just what you need
df_places = df_places[["locationid", "measureid", "data_value"]]

df_places["data_value"] = pd.to_numeric(df_places["data_value"], errors="coerce")

# Pivot to one row per county
df_places_wide = df_places.pivot_table(
    index="locationid",
    columns="measureid",
    values="data_value"
).reset_index()

# Rename the index column to geoid for merging
df_places_wide.columns.name = None
df_places_wide = df_places_wide.rename(columns={"locationid": "geoid"})
df_places_wide["geoid"] = df_places_wide["geoid"].astype(str).str.zfill(5)

In [13]:
df_places_wide.shape

(3144, 8)

In [14]:
df_places_wide = df_places_wide.rename(columns={
    "ACCESS2": "uninsured_pct",
    "BPHIGH": "high_bp_pct",
    "CASTHMA": "asthma_pct",
    "CHECKUP": "no_check_up_pct",
    "DENTAL": "no_dental_visit_pct",
    "DIABETES": "diabetes_pct",
    "HIGHCHOL": "high_cholesterol_pct"
})

In [15]:
df_places_wide.to_csv("cdc_prevalence.csv", index=False)

In [68]:
# Tract-level endpoint
url = "https://data.cdc.gov/resource/cwsq-ngmh.json"

all_data = []
limit = 50000
offset = 0

while True:
    params = {"$limit": limit, "$offset": offset}
    response = requests.get(url, params=params)
    batch = response.json()
    
    if not batch:
        break
    
    all_data.extend(batch)
    offset += limit
    print(f"Downloaded {offset} rows...")

df_places_tract = pd.DataFrame(all_data)

Downloaded 50000 rows...
Downloaded 100000 rows...
Downloaded 150000 rows...
Downloaded 200000 rows...
Downloaded 250000 rows...
Downloaded 300000 rows...
Downloaded 350000 rows...
Downloaded 400000 rows...
Downloaded 450000 rows...
Downloaded 500000 rows...
Downloaded 550000 rows...
Downloaded 600000 rows...
Downloaded 650000 rows...
Downloaded 700000 rows...
Downloaded 750000 rows...
Downloaded 800000 rows...
Downloaded 850000 rows...
Downloaded 900000 rows...
Downloaded 950000 rows...
Downloaded 1000000 rows...
Downloaded 1050000 rows...
Downloaded 1100000 rows...
Downloaded 1150000 rows...
Downloaded 1200000 rows...
Downloaded 1250000 rows...
Downloaded 1300000 rows...
Downloaded 1350000 rows...
Downloaded 1400000 rows...
Downloaded 1450000 rows...
Downloaded 1500000 rows...
Downloaded 1550000 rows...
Downloaded 1600000 rows...
Downloaded 1650000 rows...
Downloaded 1700000 rows...
Downloaded 1750000 rows...
Downloaded 1800000 rows...
Downloaded 1850000 rows...
Downloaded 1900000 ro

In [78]:
df_places_tract.to_csv("cdc_prevalence_tractlvl_raw.csv", index=False)

In [75]:
df_places_tract.head(5)

,year,stateabbr,statedesc,countyname,countyfips,locationname,datasource,category,measure,data_value_unit,...,high_confidence_limit,totalpopulation,totalpop18plus,geolocation,locationid,categoryid,measureid,datavaluetypeid,short_question_text,:@computed_region_skr5_azej
0,2023,AL,Alabama,Jefferson,01073,01073010900,BRFSS,Disability,Any disability among adults,%,...,49.0,4719,3410,"{'type': 'Point', 'coordinates': [-86.7705771,...",01073010900,DISABLT,DISABILITY,CrdPrv,Any Disability,1583
1,2023,AL,Alabama,Jefferson,01073,01073011207,BRFSS,Health Outcomes,Depression among adults,%,...,25.9,5103,3684,"{'type': 'Point', 'coordinates': [-86.6742325,...",01073011207,HLTHOUT,DEPRESSION,CrdPrv,Depression,1583
2,2023,AL,Alabama,Lauderdale,01077,01077010100,BRFSS,Disability,Any disability among adults,%,...,46.2,2278,2093,"{'type': 'Point', 'coordinates': [-87.6598801,...",01077010100,DISABLT,DISABILITY,CrdPrv,Any Disability,1584
3,2023,AL,Alabama,Lee,01081,01081040202,BRFSS,Health Outcomes,Arthritis among adults,%,...,26.2,3084,2459,"{'type': 'Point', 'coordinates': [-85.4521355,...",01081040202,HLTHOUT,ARTHRITIS,CrdPrv,Arthritis,1586
4,2023,AL,Alabama,Lee,01081,01081040300,BRFSS,Health Outcomes,Obesity among adults,%,...,42.2,2576,2188,"{'type': 'Point', 'coordinates': [-85.4668576,...",01081040300,HLTHOUT,OBESITY,CrdPrv,Obesity,1586


In [79]:
measures = [
    "ACCESS2",   # lack of health insurance (redundant with SAHIE but good validation)
    "DIABETES",  # diabetes prevalence
    "HIGHCHOL",  # high cholesterol
    "BPHIGH",    # high blood pressure
    "CASTHMA",   # asthma
    "CHECKUP",   # no routine checkup in past year — very relevant for care desert validation
    "DENTAL",    # no dental visit
]

df_places_tract = df_places_tract[df_places_tract["measureid"].isin(measures)]

In [80]:
# Keep what you need
df_places_tract = df_places_tract[["locationid", "measureid", "data_value"]]

df_places_tract["data_value"] = pd.to_numeric(df_places_tract["data_value"], errors="coerce")

# Pivot wide
df_places_tract_wide = df_places_tract.pivot_table(
    index="locationid",
    columns="measureid",
    values="data_value"
).reset_index()

df_places_tract_wide.columns.name = None
df_places_tract_wide = df_places_tract_wide.rename(columns={"locationid": "geoid"})
df_places_tract_wide["geoid"] = df_places_tract_wide["geoid"].astype(str).str.zfill(11)

# Add county geoid for linking back up
df_places_tract_wide["geoid_county"] = df_places_tract_wide["geoid"].str[:5]

In [82]:
df_places_tract_wide = df_places_tract_wide.rename(columns={
    "ACCESS2": "uninsured_pct",
    "BPHIGH": "high_bp_pct",
    "CASTHMA": "asthma_pct",
    "CHECKUP": "no_check_up_pct",
    "DENTAL": "no_dental_visit_pct",
    "DIABETES": "diabetes_pct",
    "HIGHCHOL": "high_cholesterol_pct"
})

In [83]:
df_places_tract_wide.head(5)

,geoid,uninsured_pct,high_bp_pct,asthma_pct,no_check_up_pct,no_dental_visit_pct,diabetes_pct,high_cholesterol_pct,geoid_county
0,01001020100,9.7,41.4,10.2,80.1,57.3,13.3,41.9,01001
1,01001020200,10.6,45.9,10.8,81.8,56.1,15.8,39.7,01001
2,01001020300,10.6,42.9,10.7,80.6,57.7,13.9,41.8,01001
3,01001020400,7.9,42.4,9.6,81.7,61.1,12.8,43.4,01001
4,01001020501,6.2,37.2,9.3,81.3,66.3,10.2,40.5,01001


In [84]:
df_places_tract_wide.to_csv("cdc_prevalence_tractlvl.csv", index=False)

In [21]:


import requests
import pandas as pd

# Tract-level endpoint
url = "https://data.cdc.gov/resource/shc3-fzig.json"

all_data = []
limit = 50000
offset = 0

while True:
    params = {"$limit": limit, "$offset": offset}
    response = requests.get(url, params=params)
    batch = response.json()
    
    if not batch:
        break
    
    all_data.extend(batch)
    offset += limit
    print(f"Downloaded {offset} rows...")

df_places_tract_2022 = pd.DataFrame(all_data)

Downloaded 50000 rows...
Downloaded 100000 rows...


In [22]:
print(df_places_tract_2022.shape)
print(df_places_tract_2022.count().isna())

(72337, 67)
stateabbr              False
statedesc              False
countyname             False
countyfips             False
tractfips              False
                       ...  
stroke_crudeprev       False
stroke_crude95ci       False
teethlost_crudeprev    False
teethlost_crude95ci    False
geolocation            False
Length: 67, dtype: bool


In [23]:
df_places_tract_2022.head()

,stateabbr,statedesc,countyname,countyfips,tractfips,totalpopulation,access2_crudeprev,access2_crude95ci,arthritis_crudeprev,arthritis_crude95ci,...,obesity_crude95ci,phlth_crudeprev,phlth_crude95ci,sleep_crudeprev,sleep_crude95ci,stroke_crudeprev,stroke_crude95ci,teethlost_crudeprev,teethlost_crude95ci,geolocation
0,FL,Florida,Pinellas,12103,12103027605,1669,11.9,"( 9.3, 15.1)",31.2,"(29.2, 33.1)",...,"(23.1, 26.4)",10.6,"( 9.1, 12.2)",27.8,"(26.0, 29.5)",3.6,"( 3.1, 4.3)",9.2,"( 5.3, 14.4)","{'type': 'Point', 'coordinates': [-82.84073978..."
1,CO,Colorado,Adams,08001,08001009001,4645,25.5,"(22.1, 28.9)",19.6,"(18.9, 20.4)",...,"(32.1, 34.0)",11.0,"(10.1, 11.9)",31.9,"(30.9, 32.8)",2.6,"( 2.4, 2.9)",17.0,"(12.3, 21.9)","{'type': 'Point', 'coordinates': [-104.9707911..."
2,CA,California,Santa Clara,06085,06085506804,3629,5.5,"( 4.7, 6.5)",20.0,"(19.1, 20.9)",...,"(21.1, 23.0)",7.4,"( 6.7, 8.1)",25.2,"(24.2, 26.2)",2.2,"( 2.0, 2.4)",5.7,"( 3.7, 8.3)","{'type': 'Point', 'coordinates': [-121.9339855..."
3,CA,California,Ventura,06111,06111008302,5380,10.3,"( 8.8, 12.0)",21.3,"(20.2, 22.4)",...,"(26.4, 29.1)",9.4,"( 8.5, 10.4)",29.6,"(28.2, 30.9)",2.6,"( 2.3, 2.9)",9.3,"( 6.1, 13.6)","{'type': 'Point', 'coordinates': [-118.6730418..."
4,CO,Colorado,El Paso,08041,08041006200,4669,23.5,"(20.5, 26.8)",22.4,"(21.7, 23.1)",...,"(30.8, 32.6)",12.0,"(11.1, 12.9)",34.0,"(33.0, 34.9)",3.2,"( 2.9, 3.6)",20.1,"(14.3, 26.1)","{'type': 'Point', 'coordinates': [-104.7346685..."


In [24]:
df_places_tract_2022.to_csv("cdc_prevalence_tractlvl_raw_2022.csv", index=False)

In [28]:
df_places_tract_2022.columns.tolist()

['stateabbr',
 'statedesc',
 'countyname',
 'countyfips',
 'tractfips',
 'totalpopulation',
 'access2_crudeprev',
 'access2_crude95ci',
 'arthritis_crudeprev',
 'arthritis_crude95ci',
 'binge_crudeprev',
 'binge_crude95ci',
 'bphigh_crudeprev',
 'bphigh_crude95ci',
 'bpmed_crudeprev',
 'bpmed_crude95ci',
 'cancer_crudeprev',
 'cancer_crude95ci',
 'casthma_crudeprev',
 'casthma_crude95ci',
 'cervical_crudeprev',
 'cervical_crude95ci',
 'chd_crudeprev',
 'chd_crude95ci',
 'checkup_crudeprev',
 'checkup_crude95ci',
 'cholscreen_crudeprev',
 'cholscreen_crude95ci',
 'colon_screen_crudeprev',
 'colon_screen_crude95ci',
 'copd_crudeprev',
 'copd_crude95ci',
 'corem_crudeprev',
 'corem_crude95ci',
 'corew_crudeprev',
 'corew_crude95ci',
 'csmoking_crudeprev',
 'csmoking_crude95ci',
 'dental_crudeprev',
 'dental_crude95ci',
 'depression_crudeprev',
 'depression_crude95ci',
 'diabetes_crudeprev',
 'diabetes_crude95ci',
 'ghlth_crudeprev',
 'ghlth_crude95ci',
 'highchol_crudeprev',
 'highchol_cr

In [74]:
df_places_tract_2022 = pd.read_csv("cdc_prevalence_tractlvl_raw_2022.csv")

In [75]:
measures = {
    "access2_crudeprev":  "uninsured_pct",
    "bphigh_crudeprev":   "high_bp_pct",
    "casthma_crudeprev":  "asthma_pct",
    "checkup_crudeprev":  "no_checkup_pct",
    "dental_crudeprev":   "no_dental_visit_pct",
    "diabetes_crudeprev": "diabetes_pct",
    "highchol_crudeprev": "high_cholesterol_pct"
}

# keep tract identifier + the measure columns, then rename
df_places_tract_2022 = (
    df_places_tract_2022[["tractfips"] + list(measures.keys())]
    .rename(columns=measures)
)

df_places_tract_2022 = df_places_tract_2022.rename(columns={"tractfips": "geoid"})
df_places_tract_2022["geoid"] = df_places_tract_2022["geoid"].astype(str).str.zfill(11)

# Add county geoid for linking back up
df_places_tract_2022["geoid_county"] = df_places_tract_2022["geoid"].str[:5]



In [78]:
df_places_tract_2022 = df_places_tract_2022.sort_values("geoid").reset_index()

In [79]:
df_places_tract_2022.head()

,index,geoid,uninsured_pct,high_bp_pct,asthma_pct,no_checkup_pct,no_dental_visit_pct,diabetes_pct,high_cholesterol_pct,geoid_county
0,226,01001020100,15.0,38.8,9.8,74.1,63.4,10.7,35.6,01001
1,2109,01001020200,19.9,43.6,10.9,76.5,55.6,13.4,32.7,01001
2,3955,01001020300,16.7,39.6,10.1,74.3,61.1,11.3,35.0,01001
3,5732,01001020400,12.5,39.2,8.8,75.5,70.3,10.2,37.7,01001
4,8015,01001020500,12.8,34.6,9.2,73.5,68.4,8.7,32.7,01001


In [80]:
df_places_tract_2022.to_csv("cdc_prevalence_tractlvl_2022.csv", index=False)

In [44]:
import pandas as pd

df_ahrf = pd.read_csv("AHRF2025.csv", low_memory=False)

In [45]:
df_ahrf.shape

(3235, 4352)

In [46]:
keep = ['fips_st_cnty'] + [col for col in df_ahrf.columns if col.endswith('_22')]
df_ahrf = df_ahrf[keep]

In [47]:
df_ahrf = df_ahrf.rename(columns={"fips_st_cnty": "geoid"})
df_ahrf["geoid"] = df_ahrf["geoid"].astype(str).str.zfill(5)



In [48]:
df_ahrf.head()

,geoid,phys_nf_prim_care_pc_exc_rsdt_22,phys_nf_prim_care_pc_rsdnt_22,md_nf_prim_care_pc_excl_rsdnt_22,md_nf_prim_care_pc_rsdnt_22,do_nf_prim_care_pc_excl_rsdnt_22,do_nf_prim_care_pc_rsdnt_22,md_nf_fed_activ_22,md_nf_activ_22,md_nf_fed_22,...,wkers_wk_metro_city_ge16_22,wkers_wk_micro_city_ge16_22,occd_hous_acs_22,occd_hous_gt_1_per_rm_pct_22,medn_home_value_22,medn_gross_rent_22,occd_hous_no_fuel_used_22,occd_hous_phone_svc_22,occd_hous_no_fuel_used_pct_22,occd_hous_phone_svc_pct_22
0,01001,25.0,1.0,24.0,1.0,1.0,0.0,45.0,43.0,52.0,...,0.0,0.0,22308.0,1.4,191800.0,1199.0,88.0,22157.0,0.4,99.3
1,01003,160.0,55.0,144.0,41.0,16.0,14.0,505.0,494.0,628.0,...,22474.0,0.0,90802.0,2.0,266000.0,1160.0,648.0,89832.0,0.7,98.9
2,01005,8.0,0.0,7.0,0.0,1.0,0.0,11.0,11.0,11.0,...,0.0,3314.0,9016.0,3.8,102700.0,627.0,57.0,8637.0,0.6,95.8
3,01007,21.0,17.0,18.0,12.0,3.0,5.0,33.0,33.0,33.0,...,0.0,0.0,7216.0,1.3,120100.0,765.0,48.0,7091.0,0.7,98.3
4,01009,12.0,0.0,12.0,0.0,0.0,0.0,18.0,18.0,21.0,...,0.0,0.0,21626.0,2.4,159800.0,734.0,112.0,21338.0,0.5,98.7


In [49]:
df_ahrf.to_csv("AHRF_2022.csv", index=False)

In [50]:
ahrf_cols = {
    "geoid":                              "geoid",
    "md_nf_prim_care_pc_excl_rsdnt_22":  "md_primary_care_count",
    "do_nf_prim_care_pc_excl_rsdnt_22":  "do_primary_care_count",
    "popn_22":                            "total_population",
}

df_ahrf = (
    df_ahrf[list(ahrf_cols.keys())]
    .rename(columns=ahrf_cols)
)

# Normalize geoid to 5-digit string
df_ahrf["geoid"] = df_ahrf["geoid"].astype(str).str.zfill(5)

# Convert counts to numeric (government files love storing numbers as strings)
count_cols = ["md_primary_care_count", "do_primary_care_count", "total_population"]
df_ahrf[count_cols] = df_ahrf[count_cols].apply(pd.to_numeric, errors="coerce")

# Combined PCP count (MDs + DOs, excl. residents)
df_ahrf["pcp_count"] = df_ahrf["md_primary_care_count"] + df_ahrf["do_primary_care_count"]

# Providers per 100k population
df_ahrf["pcp_per_100k"] = (df_ahrf["pcp_count"] / df_ahrf["total_population"]) * 100_000

df_ahrf_renamed = df_ahrf.sort_values("geoid")


In [51]:

print(df_ahrf_renamed.shape)
print(df_ahrf_renamed.head())
print(df_ahrf_renamed["pcp_per_100k"].describe())

(3235, 6)
   geoid  md_primary_care_count  do_primary_care_count  total_population  \
0  01001                   24.0                    1.0           59759.0   
1  01003                  144.0                   16.0          246435.0   
2  01005                    7.0                    1.0           24706.0   
3  01007                   18.0                    3.0           22005.0   
4  01009                   12.0                    0.0           59512.0   

   pcp_count  pcp_per_100k  
0       25.0     41.834703  
1      160.0     64.925843  
2        8.0     32.380798  
3       21.0     95.432856  
4       12.0     20.164001  
count    3114.000000
mean       50.546354
std        37.599820
min         0.000000
25%        25.740488
50%        44.407556
75%        68.753036
max       624.619977
Name: pcp_per_100k, dtype: float64


In [52]:
df_ahrf_renamed.to_csv("AHRF_2022_renamed.csv", index=False)